# O.G.R.E. Fine-Tuning: Qwen3.5 9B Statistics Grader

Fine-tunes Qwen3.5 9B with QLoRA on 283 statistics grading examples.

**Pipeline:** Upload JSONL → Install Unsloth → Load model (4-bit) → Train (QLoRA) → Export GGUF → Download

**Requirements:** Colab with GPU runtime (T4 16GB free tier works, A100 40GB is faster)

---

## 0. GPU Check
Verify you have a GPU runtime. Training Qwen3.5-9B at bf16 requires **~22GB VRAM minimum**.

Go to **Runtime → Change runtime type → A100 GPU** (Colab Pro required). T4 (16GB) is too small for full bf16.

In [ ]:
!nvidia-smi
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Go to Runtime > Change runtime type > A100 GPU")

vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
if vram < 20:
    print(f"WARNING: Only {vram:.1f} GB VRAM detected. Qwen3.5-9B at bf16 needs ~22GB. Use A100 (Colab Pro).")
else:
    print(f"VRAM OK: {vram:.1f} GB")

## 1. Install Unsloth + Dependencies
This takes ~2-3 minutes on a fresh Colab runtime.

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir --no-deps unsloth unsloth_zoo

## 2. Upload Training Data

Upload your two JSONL files:
- `finetune-grading.jsonl` (283 training entries)
- `finetune-grading-val.jsonl` (46 validation entries)

**Option A:** Run the cell below to use the upload dialog.  
**Option B:** Drag files into the Colab file browser (left sidebar).

In [ ]:
from google.colab import files
import os

# Upload training data
print("Upload finetune-grading.jsonl (training) and finetune-grading-val.jsonl (validation)")
uploaded = files.upload()

for fn in uploaded:
    print(f"  Uploaded: {fn} ({len(uploaded[fn]):,} bytes)")

# Verify files exist
assert os.path.exists("finetune-grading.jsonl"), "Missing training file!"
print(f"\nTraining file: finetune-grading.jsonl")

if os.path.exists("finetune-grading-val.jsonl"):
    print(f"Validation file: finetune-grading-val.jsonl")
else:
    print("No validation file uploaded (optional but recommended)")

## 3. Load & Inspect Training Data

In [ ]:
import json

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_data = load_jsonl("finetune-grading.jsonl")
print(f"Training entries: {len(train_data)}")

val_data = []
if os.path.exists("finetune-grading-val.jsonl"):
    val_data = load_jsonl("finetune-grading-val.jsonl")
    print(f"Validation entries: {len(val_data)}")

# Verify format
sample = train_data[0]
assert "messages" in sample, "Expected 'messages' key in each entry"
roles = [m["role"] for m in sample["messages"]]
assert roles == ["system", "user", "assistant"], f"Unexpected roles: {roles}"
print(f"Format: system/user/assistant messages \u2714")

# Score distribution
from collections import Counter
scores = []
for entry in train_data:
    assistant_msg = entry["messages"][-1]["content"]
    try:
        score = json.loads(assistant_msg)["score"]
        scores.append(score)
    except (json.JSONDecodeError, KeyError):
        scores.append("PARSE_ERROR")

score_counts = Counter(scores)
print(f"\nScore distribution:")
for s in sorted([k for k in score_counts if k != "PARSE_ERROR"]):
    print(f"  {s:>5}: {score_counts[s]}")
if "PARSE_ERROR" in score_counts:
    print(f"  PARSE_ERROR: {score_counts['PARSE_ERROR']} (WARNING!)")

# Token length estimate
total_chars = sum(len(m["content"]) for entry in train_data for m in entry["messages"])
print(f"\nTotal characters: {total_chars:,}")
print(f"Estimated tokens: ~{total_chars // 4:,} (rough 4 chars/token)")

## 4. Load Qwen3.5 9B (bf16 — Full Precision)

Loading at full bf16 precision. **Requires A100 (40GB) — T4 16GB is too small for a 9B model at bf16.**

Runtime → Change runtime type → A100 GPU (Colab Pro required)

In [ ]:
from unsloth import FastVisionModel

max_seq_length = 4096  # Covers our longest training examples

# FastVisionModel preserves the vision encoder for handwritten image inference
model, tokenizer = FastVisionModel.from_pretrained(
    model_name="unsloth/Qwen3.5-9B",
    max_seq_length=max_seq_length,
    load_in_4bit=False,
    load_in_16bit=True,    # bf16 LoRA — QLoRA not recommended for Qwen3.5
    full_finetuning=False,
)

print(f"\nModel loaded successfully!")
print(f"Model type: {type(model).__name__}")
print(f"Max sequence length: {max_seq_length}")

## 5. Configure LoRA Adapters

Adding LoRA adapters to the model. Only these adapter weights get trained — the base model stays frozen.

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,      # Vision encoder frozen — training on text only
    finetune_language_layers=True,     # Train the grading/language behavior
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    target_modules="all-linear",
    modules_to_save=["lm_head", "embed_tokens"],
)

# Count trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 6. Prepare Dataset

Convert our JSONL messages into the chat template format Qwen3.5 expects.

In [ ]:
from datasets import Dataset

def format_chat(example):
    """Apply the tokenizer's chat template to format messages."""
    messages = example["messages"]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

# Build training dataset
train_dataset = Dataset.from_list(train_data).map(format_chat)
print(f"Training dataset: {len(train_dataset)} examples")

# Build validation dataset (if available)
eval_dataset = None
if val_data:
    eval_dataset = Dataset.from_list(val_data).map(format_chat)
    print(f"Validation dataset: {len(eval_dataset)} examples")

# Preview one formatted example
print(f"\n--- Sample (first 500 chars) ---")
print(train_dataset[0]["text"][:500])
print("...")

# Check token lengths
token_lengths = [len(tokenizer.tokenizer.encode(ex["text"])) for ex in train_dataset]
print(f"\nToken lengths — min: {min(token_lengths)}, max: {max(token_lengths)}, "
      f"mean: {sum(token_lengths)/len(token_lengths):.0f}")
over_limit = sum(1 for l in token_lengths if l > max_seq_length)
if over_limit:
    print(f"WARNING: {over_limit} examples exceed max_seq_length={max_seq_length}! They will be truncated.")
else:
    print(f"All examples fit within max_seq_length={max_seq_length} \u2714")

## 7. Train

QLoRA fine-tuning with SFTTrainer. ~15-30 min on T4, ~5-10 min on A100.

**Key hyperparameters:**
- 2 epochs (283 examples is small — 2 epochs helps without overfitting)
- Learning rate 2e-5 with cosine schedule
- Effective batch size 8 (per_device=2 × gradient_accumulation=4)
- Warmup: 10% of total steps

In [ ]:
from trl import SFTTrainer, SFTConfig
import math

# Calculate steps for logging
effective_batch_size = 8  # per_device_batch * grad_accum = 2 * 4
steps_per_epoch = math.ceil(len(train_dataset) / effective_batch_size)
total_steps = steps_per_epoch * 2  # 2 epochs
warmup_steps = max(1, int(total_steps * 0.1))

print(f"Steps per epoch: {steps_per_epoch}")
print(f"Total steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        output_dir="./outputs",
        num_train_epochs=2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-5,
        lr_scheduler_type="cosine",
        warmup_steps=warmup_steps,
        weight_decay=0.01,
        max_grad_norm=1.0,
        optim="adamw_8bit",
        logging_steps=5,
        eval_strategy="steps" if eval_dataset else "no",
        eval_steps=steps_per_epoch if eval_dataset else None,
        save_strategy="epoch",
        seed=42,
        report_to="none",
        max_seq_length=max_seq_length,
        dataset_num_proc=2,
    ),
    packing=False,
    dataset_text_field="text",
)

print(f"\nReady to train. Starting...")

In [ ]:
# Run training
trainer_stats = trainer.train()

print(f"\n{'='*50}")
print(f"Training complete!")
print(f"Total steps: {trainer_stats.global_step}")
print(f"Training loss: {trainer_stats.training_loss:.4f}")
print(f"Runtime: {trainer_stats.metrics['train_runtime']:.0f}s "
      f"({trainer_stats.metrics['train_runtime']/60:.1f} min)")
print(f"{'='*50}")

## 8. Quick Sanity Check

Test the fine-tuned model on a sample prompt before exporting.

In [ ]:
# Switch to inference mode
FastVisionModel.for_inference(model)

# Build a simple test prompt
test_messages = [
    {"role": "system", "content": "You are an expert grading assistant. Grade this student work against the provided rubric. Output: JSON object only."},
    {"role": "user", "content": """GRADING PHILOSOPHY:
Grade each response against the rubric criteria provided.

MAX SCORE: 10

QUESTION/PROMPT:
Explain what a p-value represents in hypothesis testing.

GRADING CHECKLIST:
- Definition of p-value (5 points)
  - Probability of observing results as extreme as the data, assuming the null hypothesis is true.
- Interpretation (5 points)
  - Small p-value = evidence against the null hypothesis.

SCORING SCALE:
0: No submission or completely off-topic
5: Partial understanding, addresses some criteria
10: Excellent, thorough and accurate response

STUDENT WORK:
A p-value is the probability that you would see your data (or something more extreme) if the null hypothesis were actually true. A small p-value means the data is unlikely under the null, giving evidence to reject it.

PARTIAL CREDIT RULE: When a criterion is addressed conceptually but with minor errors, award 60-80% of that criterion's weight.

Return ONLY valid JSON. No markdown code fences. No explanation text.
{"score": <0-10>}"""}
]

# Tokenize and generate
inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=64,
    temperature=0.1,
    do_sample=True,
)

# Decode only the generated tokens
generated = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print(f"Model output: {generated}")

# Try to parse as JSON
try:
    result = json.loads(generated.strip())
    print(f"Parsed score: {result['score']}")
    if 8 <= result["score"] <= 10:
        print("Sanity check PASSED - model graded a strong response highly")
    else:
        print(f"Score {result['score']} is unexpected for this strong response. Investigate.")
except (json.JSONDecodeError, KeyError) as e:
    print(f"WARNING: Could not parse output as JSON: {e}")
    print("The model may need more training or the output format needs attention.")

## 9. Export to GGUF

Unsloth can export directly to GGUF format — no need for a separate llama.cpp step.

We'll export **Q4_K_M** (recommended for 12GB VRAM) and optionally **Q5_K_M** for higher quality.

In [ ]:
# Export Q4_K_M (primary — ~5.5GB, fits 12GB VRAM with room for KV cache)
model.save_pretrained_gguf(
    "qwen35-9b-stat-grader-q4km",
    tokenizer,
    quantization_method="q4_k_m",
)
print("\nQ4_K_M export complete!")

# Show the output file
import glob
gguf_files = glob.glob("qwen35-9b-stat-grader-q4km/*.gguf")
for f in gguf_files:
    size_gb = os.path.getsize(f) / (1024**3)
    print(f"  {f} ({size_gb:.2f} GB)")

In [ ]:
# OPTIONAL: Also export Q5_K_M (higher quality, ~6.5GB)
# Uncomment the lines below if you want a second quantization option

# model.save_pretrained_gguf(
#     "qwen35-9b-stat-grader-q5km",
#     tokenizer,
#     quantization_method="q5_k_m",
# )
# print("Q5_K_M export complete!")

## 10. Save to Google Drive & Download

Copy the GGUF to Google Drive for persistent storage, then download locally.

In [ ]:
from google.colab import drive
import shutil

# Mount Google Drive
drive.mount("/content/drive")

# Create output directory in Drive
drive_dir = "/content/drive/MyDrive/OGRE-finetune"
os.makedirs(drive_dir, exist_ok=True)

# Copy GGUF files to Drive
for f in glob.glob("qwen35-9b-stat-grader-q4km/*.gguf"):
    dest = os.path.join(drive_dir, os.path.basename(f))
    print(f"Copying {f} to Drive...")
    shutil.copy2(f, dest)
    print(f"  Saved: {dest} ({os.path.getsize(dest) / (1024**3):.2f} GB)")

# Also save the LoRA adapter (small, good backup)
model.save_pretrained(os.path.join(drive_dir, "lora-adapter"))
tokenizer.save_pretrained(os.path.join(drive_dir, "lora-adapter"))
print(f"\nLoRA adapter saved to Drive (backup)")

print(f"\nAll files saved to: {drive_dir}")
!ls -lh "{drive_dir}"

In [ ]:
# Download GGUF directly to your local machine
# (Alternative to Google Drive — use whichever is faster for you)

for f in glob.glob("qwen35-9b-stat-grader-q4km/*.gguf"):
    print(f"Downloading {f}...")
    files.download(f)
    print("Download started! Check your browser's download bar.")

## 11. Training Loss History

Plot the training loss curve to check for overfitting or instability.

In [ ]:
import matplotlib.pyplot as plt

# Extract loss from training log
log_history = trainer.state.log_history
train_losses = [(entry["step"], entry["loss"]) for entry in log_history if "loss" in entry]
eval_losses = [(entry["step"], entry["eval_loss"]) for entry in log_history if "eval_loss" in entry]

if train_losses:
    steps, losses = zip(*train_losses)
    plt.figure(figsize=(10, 4))
    plt.plot(steps, losses, label="Training Loss", alpha=0.8)
    if eval_losses:
        eval_steps, eval_loss_vals = zip(*eval_losses)
        plt.plot(eval_steps, eval_loss_vals, "ro-", label="Validation Loss", markersize=8)
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title("O.G.R.E. Fine-Tuning Loss Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("training_loss.png", dpi=100)
    plt.show()
    print(f"\nFinal training loss: {losses[-1]:.4f}")
    if eval_losses:
        print(f"Final validation loss: {eval_loss_vals[-1]:.4f}")
        if eval_loss_vals[-1] > losses[-1] * 1.5:
            print("WARNING: Validation loss is significantly higher than training loss. Possible overfitting.")
else:
    print("No training loss data found in log history.")

---

## Next Steps (Local Machine)

After downloading the GGUF file, run these commands on your local Windows machine:

```bash
# 1. Create Ollama Modelfile (save as 'Modelfile' in the same directory as the GGUF)
#    See the Modelfile in the project repo: test-data/Modelfile

# 2. Import into Ollama
ollama create qwen3.5-9B-stat-grader -f Modelfile

# 3. Test
ollama run qwen3.5-9B-stat-grader

# 4. Re-run benchmark
# (from project root)
```

The Modelfile is provided in `test-data/Modelfile` in the O.G.R.E. repository.